In [7]:
import pandas as pd
import wandb

ENTITY = "behzadshomali"
PROJECT = "loop_MTP_paper_evals"

api = wandb.Api()


def get_eval_values(run_id: str, project: str = PROJECT, entity: str = ENTITY):
    """Retrieve eval values for a given W&B run id.

    Returns:
        history: DataFrame indexed by `seen_steps`, columns are `eval/*` and `summary/*`.
        summary: dict of final `eval/*` / `summary/*` values from run.summary.
    """
    run = api.run(f"{entity}/{project}/{run_id}")

    hist = run.history(pandas=True, samples=10000)
    eval_cols = [c for c in hist.columns if c.startswith("eval/") or c.startswith("summary/") or c.startswith("eval_full/")]
    keep = (["seen_steps"] + eval_cols) if "seen_steps" in hist.columns else eval_cols
    history = hist[keep].dropna(how="all", subset=eval_cols)
    if "seen_steps" in history.columns:
        history = history.set_index("seen_steps").sort_index()

    summary = {k: v for k, v in dict(run.summary).items()
               if k.startswith("eval/") or k.startswith("summary/")}

    return history, summary


In [16]:
def bpb_acc_detail(id_bpb, id_acc=None):
    print(f"run name: {api.run(f'{ENTITY}/{PROJECT}/{id_bpb}').name}")
    if id_acc is not None:
        print(f"run name: {api.run(f'{ENTITY}/{PROJECT}/{id_acc}').name}")

    history_bpb, summary_bpb = get_eval_values(id_bpb)

    if id_acc is not None:
        history_acc, summary_acc = get_eval_values(id_acc)

    general_bpb_tasks = [
        'eval/hellaswag:rc:bpb::olmes:full', 
        'eval/piqa:rc:bpb::olmes:full', 
        'eval/lambada:bpb', 
        'eval/arc_easy:rc:bpb::olmes:full', 
        'eval/arc_challenge:rc:bpb::olmes:full', 
        'eval/winogrande:rc:bpb::olmes:full', 
        'eval/socialiqa:rc:bpb::olmes:full'
    ]

    math_bpb_tasks = [
        "eval/minerva_math_algebra:bpb::olmes",
        "eval/minerva_math_counting_and_probability:bpb::olmes",
        "eval/minerva_math_geometry:bpb::olmes",
        "eval/minerva_math_intermediate_algebra:bpb::olmes",
        "eval/minerva_math_number_theory:bpb::olmes",
        "eval/minerva_math_prealgebra:bpb::olmes",
        "eval/minerva_math_precalculus:bpb::olmes"
    ]

    code_bpb_tasks = [
        "eval/mbpp:3shot:bpb::none",
        "eval/codex_humaneval:3shot:bpb::none",
    ]

    try:
        general_bpb_values = history_bpb.dropna(subset=general_bpb_tasks)[general_bpb_tasks].iloc[-1]
        general_bpb_avg = general_bpb_values.mean()

        math_bpb_values = history_bpb.dropna(subset=math_bpb_tasks)[math_bpb_tasks].iloc[-1]
        math_bpb_avg = math_bpb_values.mean()

        code_bpb_values = history_bpb.dropna(subset=code_bpb_tasks)[code_bpb_tasks].iloc[-1]
        code_bpb_avg = code_bpb_values.mean()
    except:
        general_bpb_tasks = [t.replace("eval/", "eval_full/") for t in general_bpb_tasks]
        math_bpb_tasks = [t.replace("eval/", "eval_full/") for t in math_bpb_tasks]
        code_bpb_tasks = [t.replace("eval/", "eval_full/") for t in code_bpb_tasks]

        general_bpb_values = history_bpb.dropna(subset=general_bpb_tasks)[general_bpb_tasks].iloc[-1]
        general_bpb_avg = general_bpb_values.mean()

        math_bpb_values = history_bpb.dropna(subset=math_bpb_tasks)[math_bpb_tasks].iloc[-1]
        math_bpb_avg = math_bpb_values.mean()

        code_bpb_values = history_bpb.dropna(subset=code_bpb_tasks)[code_bpb_tasks].iloc[-1]
        code_bpb_avg = code_bpb_values.mean()

    
    for t in general_bpb_tasks:
        print(f"{general_bpb_values[t]:.4f}", end=" & ")
    print(f"{general_bpb_avg:.4f}", end= " & ")

    for t in math_bpb_tasks:
        print(f"{math_bpb_values[t]:.4f}", end=" & ")
    print(f"{math_bpb_avg:.4f}", end= " & ")

    for t in code_bpb_tasks:
        print(f"{code_bpb_values[t]:.4f}", end=" & ")
    print(f"{code_bpb_avg:.4f}")
    

In [22]:
id_bpb = "qxa4aht3"
id_acc = None

bpb_acc_detail(id_bpb=id_bpb, id_acc=id_acc)

run name: loopformer_L9_out_L9_LR0.0019_WD0.1_LRPCTSTART0.08
0.8704 & 1.1191 & 0.7707 & 0.6821 & 0.8900 & 1.2129 & 1.1735 & 0.9598 & 0.6643 & 0.6533 & 0.7445 & 0.6941 & 0.7342 & 0.6392 & 0.6273 & 0.6795 & 0.9236 & 0.7264 & 0.8250


In [21]:
RUN_ID = "t56re8me" 

history, summary = get_eval_values(RUN_ID)

print(f"history shape: {history.shape}")
# print run name and summary
print(f"run name: {api.run(f'{ENTITY}/{PROJECT}/{RUN_ID}').name}")

# show the history table with 4 digits of precision
pd.set_option("display.precision", 3)

# don't show the summary columns in the history table
eval_cols = [c for c in history.columns if c.startswith("eval/")]
history[eval_cols].tail(2)


history shape: (0, 0)
run name: SEED3812_WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.01_NormEmbdsFalse_k=12_L=5_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-07-01__03-41-40_810df85b00591a9f


""


In [22]:
# now show the summary runs
eval_cols = [c for c in history.columns if c.startswith("summary/")]
history[eval_cols].tail(2)


""


In [23]:
# now show the summary runs
pd.set_option("display.precision", 2)
eval_cols = [c for c in history.columns if c.startswith("eval_full/")]
# convert to percentage
history[eval_cols].tail(2) * 100


""


In [24]:
# TASKS = [
#     "eval/mbpp:3shot:bpb::none",
#     "eval/codex_humaneval:3shot:bpb::none",
# ]

# # print(history.dropna(subset=TASKS)[TASKS])
# values = history.dropna(subset=TASKS)[TASKS].iloc[-1]
# avg = values.mean()

# pd.set_option("display.precision", 3)
# print(values.round(3).to_string())
# print(f"\nAverage: {avg:.3f}")

In [25]:
# # Average of selected eval tasks (in %)
# TASKS = [
#     "eval_full/arc_challenge:rc::olmes:full",
#     "eval_full/arc_easy:rc::olmes:full",
#     "eval_full/hellaswag:rc::olmes:full",
#     "eval_full/winogrande:rc::olmes:full",
#     "eval_full/socialiqa:rc::olmes:full",
#     "eval_full/piqa:rc::olmes:full",
#     "eval_full/lambada",
#     "eval_full/qasper_yesno:rc::olmes",
# ]

# values = history[TASKS].iloc[-1] * 100
# avg = values.mean()

# print(values.round(2).to_string())
# print(f"\nAverage: {avg:.2f}")


In [26]:
# for v in values:
#     print(f"{v:.2f} &", end=" ")

# print(f"{avg:.2f}")

In [27]:
TASKS = [
    "summary/modalities:base_easy:qa_bpb",
    "summary/modalities:base_easy:math_bpb",
    "summary/modalities:base_easy:code_bpb",
    "summary/avg_all_suites"
]

values = history.dropna(subset=TASKS)[TASKS].iloc[-1]
print(values)
i = 0
for t in TASKS:
    v = values[t]
    if i == 1:
        print("&", end=" ")
    print(f"{v:.4f} &", end=" ")
    i += 1

KeyError: ['summary/modalities:base_easy:qa_bpb', 'summary/modalities:base_easy:math_bpb', 'summary/modalities:base_easy:code_bpb', 'summary/avg_all_suites']

In [28]:
import numpy as np
print("math", f"{np.std([0.6581, 0.6569]):.4f}")
print("code", f"{np.std([0.7967, 0.7876]):.4f}")
print("qa  ", f"{np.std([0.9408, 0.9375]):.4f}")
print("avg ", f"{np.std([0.7986, 0.7940]):.4f}")

math 0.0006
code 0.0045
qa   0.0016
avg  0.0023


In [29]:
# TASKS = [
#     "eval/arc_challenge:rc:bpb::olmes:full",
#     "eval/arc_easy:rc:bpb::olmes:full",
#     "eval/hellaswag:rc:bpb::olmes:full",
#     "eval/winogrande:rc:bpb::olmes:full",
#     "eval/socialiqa:rc:bpb::olmes:full",
#     "eval/piqa:rc:bpb::olmes:full",
#     # "eval/qasper_yesno:rc:bpb::olmes",
#     "eval/lambada:bpb",
# ]

# # print(history.dropna(subset=TASKS)[TASKS])
# values = history.dropna(subset=TASKS)[TASKS].iloc[-1]
# avg = values.mean()

# pd.set_option("display.precision", 3)
# print(values.round(3).to_string())
# print(f"\nAverage: {avg:.3f}")

In [30]:
# TASKS = [
#     "eval/minerva_math_algebra:bpb::olmes",
#     "eval/minerva_math_counting_and_probability:bpb::olmes",
#     "eval/minerva_math_geometry:bpb::olmes",
#     "eval/minerva_math_intermediate_algebra:bpb::olmes",
#     "eval/minerva_math_number_theory:bpb::olmes",
#     "eval/minerva_math_prealgebra:bpb::olmes",
#     "eval/minerva_math_precalculus:bpb::olmes",
# ]

# values = history.dropna(subset=TASKS)[TASKS].iloc[-1]
# avg = values.mean()
# pd.set_option("display.precision", 3)
# print(values.round(3).to_string())
# print(f"\nAverage: {avg:.3f}")

In [31]:
TASKS = [
    "eval_full/arc_challenge:rc::olmes:full",
    "eval_full/arc_easy:rc::olmes:full",
    "eval_full/hellaswag:rc::olmes:full",
    "eval_full/winogrande:rc::olmes:full",
    "eval_full/socialiqa:rc::olmes:full",
    "eval_full/piqa:rc::olmes:full",
    "eval_full/lambada",
    # "eval_full/qasper_yesno:rc::olmes",
]

In [32]:
for t in TASKS:
    print("${", "summary:", t, "}", end="+", sep="")

${summary:eval_full/arc_challenge:rc::olmes:full}+${summary:eval_full/arc_easy:rc::olmes:full}+${summary:eval_full/hellaswag:rc::olmes:full}+${summary:eval_full/winogrande:rc::olmes:full}+${summary:eval_full/socialiqa:rc::olmes:full}+${summary:eval_full/piqa:rc::olmes:full}+${summary:eval_full/lambada}+

In [ ]:
TASKS = {
    "eval_full/arc_challenge:rc::olmes:full": 0.3455631399317406,
    "eval_full/arc_easy:rc::olmes:full": 0.6346801346801347,
    "eval_full/hellaswag:rc::olmes:full",
    "eval_full/winogrande:rc::olmes:full",
    "eval_full/socialiqa:rc::olmes:full",
    "eval_full/piqa:rc::olmes:full",
    "eval_full/lambada",
    # "eval_full/qasper_yesno:rc::olmes",
}

In [33]:
values = history.dropna(subset=TASKS)[TASKS].iloc[-1] * 100
avg = values.mean()
pd.set_option("display.precision", 3)
print(values.round(2).to_string())
print(f"\nAverage: {avg:.2f}")

KeyError: ['eval_full/arc_challenge:rc::olmes:full', 'eval_full/arc_easy:rc::olmes:full', 'eval_full/hellaswag:rc::olmes:full', 'eval_full/winogrande:rc::olmes:full', 'eval_full/socialiqa:rc::olmes:full', 'eval_full/piqa:rc::olmes:full', 'eval_full/lambada']

In [14]:
# # for every run with this pattern: ^WD0.1.*AlignedHidden0\.\d.*L=3.
# # run the following and sort them based on the average of these tasks:

# TASKS = [
#     "eval_full/arc_challenge:rc::olmes:full",
#     "eval_full/arc_easy:rc::olmes:full",
#     "eval_full/hellaswag:rc::olmes:full",
#     "eval_full/winogrande:rc::olmes:full",
#     "eval_full/socialiqa:rc::olmes:full",
#     "eval_full/piqa:rc::olmes:full",
#     "eval_full/lambada",
#     # "eval_full/qasper_yesno:rc::olmes",
# ]
# values = history.dropna(subset=TASKS)[TASKS].iloc[-1] * 100
# avg = values.mean()
# pd.set_option("display.precision", 3)
# print(values.round(2).to_string())
# print(f"\nAverage: {avg:.2f}")

In [238]:
import re

# PATTERN = r"^WD0.1.*AlignedHidden0\.\d.*L=3."
iter = 5
PATTERN = f"^loopformer_L{iter}_out_L{iter}"

TASKS = [
    "eval_full/arc_challenge:rc::olmes:full",
    "eval_full/arc_easy:rc::olmes:full",
    "eval_full/hellaswag:rc::olmes:full",
    "eval_full/winogrande:rc::olmes:full",
    "eval_full/socialiqa:rc::olmes:full",
    "eval_full/piqa:rc::olmes:full",
    "eval_full/lambada",
    # "eval_full/qasper_yesno:rc::olmes",
]

# retrieve all runs in the project and keep those whose name matches the pattern
matching = [r for r in api.runs(f"{ENTITY}/{PROJECT}") if re.search(PATTERN, r.name)]
print(f"matched {len(matching)} runs")

rows = {}
for run in matching:
    hist, _ = get_eval_values(run.id)
    present = [t for t in TASKS if t in hist.columns]
    if not present:
        print(f"  [skip] {run.name}: no matching task columns")
        continue
    sub = hist.dropna(subset=present)
    if sub.empty:
        print(f"  [skip] {run.name}: no rows with task values")
        continue
    vals = sub[present].iloc[-1] * 100
    rows[run.name] = vals

table = pd.DataFrame(rows).T
table["avg"] = table.mean(axis=1)
table = table.sort_values("avg", ascending=False)

pd.set_option("display.precision", 2)
table


matched 16 runs
  [skip] loopformer_L5_out_L5_LR0.0004_WD0.1_LRPCTSTART0.001: no matching task columns
  [skip] loopformer_L5_out_L5_LR0.0004_WD0.1_LRPCTSTART0.08: no matching task columns
  [skip] loopformer_L5_out_L5_LR0.0004_WD0.2_LRPCTSTART0.001: no matching task columns
  [skip] loopformer_L5_out_L5_LR0.0013_WD0.1_LRPCTSTART0.001: no matching task columns
  [skip] loopformer_L5_out_L5_LR0.0004_WD0.2_LRPCTSTART0.08: no matching task columns


,eval_full/arc_challenge:rc::olmes:full,eval_full/arc_easy:rc::olmes:full,eval_full/hellaswag:rc::olmes:full,eval_full/winogrande:rc::olmes:full,eval_full/socialiqa:rc::olmes:full,eval_full/piqa:rc::olmes:full,eval_full/lambada,avg
loopformer_L5_out_L5_LR0.0019_WD0.2_LRPCTSTART0.001,34.47,61.03,42.18,53.43,46.01,65.94,34.78,48.26
loopformer_L5_out_L5_LR0.0019_WD0.2_LRPCTSTART0.08,33.45,61.36,42.09,52.80,45.04,66.54,34.14,47.92
loopformer_L5_out_L5_LR0.0006_WD0.2_LRPCTSTART0.001,32.51,60.31,40.29,52.01,45.24,64.91,33.46,46.96
loopformer_L5_out_L5_LR0.0038_WD0.2_LRPCTSTART0.08,34.56,59.05,41.24,51.62,44.11,65.51,32.16,46.89
loopformer_L5_out_L5_LR0.0006_WD0.1_LRPCTSTART0.001,33.53,60.90,39.28,51.22,45.04,65.40,32.41,46.82
loopformer_L5_out_L5_LR0.0006_WD0.1_LRPCTSTART0.08,33.45,59.34,39.23,52.25,42.78,65.18,32.33,46.37
loopformer_L5_out_L5_LR0.0006_WD0.2_LRPCTSTART0.08,32.76,59.47,38.53,49.49,46.47,65.23,31.13,46.15
loopformer_L5_out_L5_LR0.0038_WD0.1_LRPCTSTART0.001,29.01,48.23,32.37,50.83,40.69,62.24,18.13,40.21
loopformer_L5_out_L5_LR0.0038_WD0.1_LRPCTSTART0.08,25.09,29.67,25.07,50.12,38.23,50.98,0.00,31.31
loopformer_L5_out_L5_LR0.0019_WD0.1_LRPCTSTART0.001,24.32,26.64,25.72,49.72,37.92,49.95,0.00,30.61


In [47]:
SUITE_KEY = "summary/avg_all_suites"

summary_rows = {}
for run in matching:
    s = {k: v for k, v in dict(run.summary).items() if k.startswith("summary/")}
    if not s:
        print(f"  [skip] {run.name}: no summary/ metrics")
        continue
    summary_rows[run.name] = s

summary_table = pd.DataFrame(summary_rows).T
# sort by avg_all_suites if present, else by the first column
sort_key = SUITE_KEY if SUITE_KEY in summary_table.columns else summary_table.columns[0]
summary_table = summary_table.sort_values(sort_key, ascending=True)

pd.set_option("display.precision", 3)
summary_table


  [skip] WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-07__07-37-14_abb524b3fc3370e2: no summary/ metrics
  [skip] WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-07__07-37-14_abb524b3fc3370e2: no summary/ metrics
  [skip] WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-07__07-37-14_abb524b3fc3370e2: no summary/ metrics
  [skip] WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-07__07-37-14_abb524b3fc3370e2: no summary/ metrics
  [skip] WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.1_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-26__16-42-50_606d54402e632e0c: no summary/ metrics
  [skip] WD0.1_12Block_[0.5, -3.0, 

,summary/avg_all_suites,summary/avg_all_suites_last,summary/modalities:base_easy:code_bpb,summary/modalities:base_easy:math_bpb,summary/modalities:base_easy:qa_bpb,summary/table
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.01_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-27__21-41-35_edc0621911083e45",0.807,0.807,0.796,0.669,0.955,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.1_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-26__16-42-50_606d54402e632e0c",0.81,0.81,0.8,0.675,0.956,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.15_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-27__21-46-43_78cfe97add77c897",0.815,0.815,0.816,0.676,0.954,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__01-14-47_f0e124cd553968ff",0.817,0.817,0.813,0.68,0.957,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.2_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__01-08-28_7b237eb4fb7f21ea",0.818,0.818,0.821,0.678,0.955,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-07__07-37-14_abb524b3fc3370e2",0.819,0.819,0.819,0.675,0.962,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.1_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-27__21-40-44_27f328e9b97eefd4",0.82,0.82,0.83,0.673,0.956,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.05_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-27__21-44-50_46e7575553cb67c7",0.82,0.82,0.836,0.672,0.953,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.5_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-27__21-44-00_3587083dea21f19c",0.824,0.824,0.829,0.685,0.959,{'_latest_artifact_path': 'wandb-client-artifa...
"WD0.1_12Block_[0.5, -3.0, -3.0]_AlignedHidden0.4_NormEmbdsFalse_k=12_L=3_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__01-40-25_c57c9f7b4252055e",0.826,0.826,0.831,0.68,0.966,{'_latest_artifact_path': 'wandb-client-artifa...


In [268]:
iter = 9
LOOP_PATTERN = f"^loopformer_L{iter}_out_L{iter}"

# retrieve all runs whose name matches the loopformer pattern
loop_matching = [r for r in api.runs(f"{ENTITY}/{PROJECT}") if re.search(LOOP_PATTERN, r.name)]
print(f"matched {len(loop_matching)} loopformer runs")

loop_rows = {}
bpb_tasks = set()
for run in loop_matching:
    hist, _ = get_eval_values(run.id)
    # eval_full/* tasks that are bpb metrics
    cols = [c for c in hist.columns if c.startswith("eval_full/") and "bpb" in c]
    if not cols:
        print(f"  [skip] {run.name}: no eval_full/*bpb columns")
        continue
    sub = hist.dropna(subset=cols, how="all")
    if sub.empty:
        print(f"  [skip] {run.name}: no rows with bpb values")
        continue
    loop_rows[run.name] = sub[cols].iloc[-1]
    bpb_tasks.update(cols)

bpb_tasks = sorted(bpb_tasks)
print(f"\n{len(bpb_tasks)} bpb tasks:")
for t in bpb_tasks:
    print(f"  {t}")

loop_table = pd.DataFrame(loop_rows).T
loop_table = loop_table.apply(pd.to_numeric, errors="coerce")
loop_table["avg"] = loop_table.mean(axis=1)
loop_table = loop_table.sort_values("avg", ascending=True)

pd.set_option("display.precision", 4)
loop_table


matched 15 loopformer runs
  [skip] loopformer_L9_out_L9_LR0.0004_WD0.1_LRPCTSTART0.08: no eval_full/*bpb columns
  [skip] loopformer_L9_out_L9_LR0.0004_WD0.2_LRPCTSTART0.001: no eval_full/*bpb columns
  [skip] loopformer_L9_out_L9_LR0.0004_WD0.1_LRPCTSTART0.001: no eval_full/*bpb columns
  [skip] loopformer_L9_out_L9_LR0.0004_WD0.2_LRPCTSTART0.08: no eval_full/*bpb columns

19 bpb tasks:
  eval_full/arc_challenge:rc:bpb::olmes:full
  eval_full/arc_easy:rc:bpb::olmes:full
  eval_full/codex_humaneval:3shot:bpb::none
  eval_full/hellaswag:rc:bpb::olmes:full
  eval_full/lambada:bpb
  eval_full/mbpp:3shot:bpb::none
  eval_full/minerva_math_algebra:bpb::olmes
  eval_full/minerva_math_counting_and_probability:bpb::olmes
  eval_full/minerva_math_geometry:bpb::olmes
  eval_full/minerva_math_intermediate_algebra:bpb::olmes
  eval_full/minerva_math_number_theory:bpb::olmes
  eval_full/minerva_math_prealgebra:bpb::olmes
  eval_full/minerva_math_precalculus:bpb::olmes
  eval_full/modalities:base_e

,eval_full/arc_challenge:rc:bpb::olmes:full,eval_full/arc_easy:rc:bpb::olmes:full,eval_full/codex_humaneval:3shot:bpb::none,eval_full/hellaswag:rc:bpb::olmes:full,eval_full/lambada:bpb,eval_full/mbpp:3shot:bpb::none,eval_full/minerva_math_algebra:bpb::olmes,eval_full/minerva_math_counting_and_probability:bpb::olmes,eval_full/minerva_math_geometry:bpb::olmes,eval_full/minerva_math_intermediate_algebra:bpb::olmes,eval_full/minerva_math_number_theory:bpb::olmes,eval_full/minerva_math_prealgebra:bpb::olmes,eval_full/minerva_math_precalculus:bpb::olmes,eval_full/modalities:base_easy:code_bpb,eval_full/modalities:base_easy:math_bpb,eval_full/olmo3:base_easy:math_bpb,eval_full/piqa:rc:bpb::olmes:full,eval_full/socialiqa:rc:bpb::olmes:full,eval_full/winogrande:rc:bpb::olmes:full,avg
loopformer_L9_out_L9_LR0.0019_WD0.1_LRPCTSTART0.08,0.8900,0.6821,0.7264,0.8704,0.7707,0.9236,0.6643,0.6533,0.7445,0.6941,0.7342,0.6392,0.6273,0.8250,0.6795,0.6795,1.1191,1.1735,1.2129,0.8058
loopformer_L9_out_L9_LR0.0019_WD0.2_LRPCTSTART0.001,0.8971,0.6958,0.7312,0.8693,0.7838,0.9219,0.6704,0.6579,0.7475,0.6983,0.7394,0.6416,0.6306,0.8266,0.6837,0.6837,1.1312,1.1698,1.2093,0.8099
loopformer_L9_out_L9_LR0.0038_WD0.2_LRPCTSTART0.001,0.9118,0.7067,0.7476,0.8783,0.8106,0.9383,0.6806,0.6645,0.7589,0.7171,0.7434,0.6469,0.6531,0.8429,0.6949,0.6949,1.1372,1.1514,1.2240,0.8212
loopformer_L9_out_L9_LR0.0006_WD0.2_LRPCTSTART0.001,0.9067,0.7023,0.7418,0.8799,0.7989,0.9681,0.6825,0.6641,0.7592,0.7129,0.7490,0.6540,0.6421,0.8549,0.6948,0.6948,1.1332,1.1794,1.2213,0.8232
loopformer_L9_out_L9_LR0.0006_WD0.1_LRPCTSTART0.001,0.9112,0.7125,0.7322,0.8819,0.8030,0.9595,0.6945,0.6690,0.7634,0.7200,0.7591,0.6635,0.6491,0.8459,0.7027,0.7027,1.1390,1.1684,1.2215,0.8263
loopformer_L9_out_L9_LR0.0006_WD0.2_LRPCTSTART0.08,0.9160,0.7121,0.7518,0.8802,0.8140,0.9447,0.6875,0.6653,0.7603,0.7136,0.7550,0.6573,0.6458,0.8482,0.6978,0.6978,1.1443,1.1705,1.2373,0.8263
loopformer_L9_out_L9_LR0.0038_WD0.2_LRPCTSTART0.08,0.9145,0.7163,0.7993,0.8807,0.7966,0.9258,0.6838,0.6683,0.7694,0.7247,0.7511,0.6552,0.6521,0.8625,0.7007,0.7007,1.1675,1.1453,1.2311,0.8287
loopformer_L9_out_L9_LR0.0006_WD0.1_LRPCTSTART0.08,0.9082,0.7114,0.7638,0.8769,0.8035,0.9495,0.6899,0.6716,0.7644,0.7215,0.7597,0.6609,0.6520,0.8567,0.7029,0.7029,1.1378,1.1887,1.2379,0.8295
loopformer_L9_out_L9_LR0.0038_WD0.1_LRPCTSTART0.001,2.0723,1.9667,2.9424,1.7109,2.8727,3.2767,2.3923,2.0848,2.6515,2.5798,2.3500,2.2318,2.5099,3.1095,2.4000,2.4000,2.0043,2.1206,2.0610,2.4072
loopformer_L9_out_L9_LR0.0038_WD0.1_LRPCTSTART0.08,2.5176,2.4510,4.8626,2.2982,3.2077,5.1003,4.4210,3.4181,4.3718,4.8083,4.0451,3.7969,4.9915,4.9814,4.2647,4.2647,2.5587,2.3908,2.4650,3.7482


In [21]:
table

,eval_full/arc_challenge:rc::olmes:full,eval_full/arc_easy:rc::olmes:full,eval_full/hellaswag:rc::olmes:full,eval_full/winogrande:rc::olmes:full,eval_full/socialiqa:rc::olmes:full,eval_full/piqa:rc::olmes:full,eval_full/lambada,avg
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.15_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-06-01__10-45-07_d85441c844f8f306",36.860,62.753,43.766,53.907,47.390,67.682,35.824,49.740
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-08__02-07-35_661065fc2f4f796f",34.215,63.594,43.955,54.301,46.418,67.900,35.610,49.428
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.5_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__16-03-33_0bd5a608f1e2f8d2",34.642,63.594,44.045,52.802,45.138,68.770,36.309,49.329
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.01_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-06-01__10-36-09_382702e94560bb41",33.362,63.173,43.955,53.986,45.803,67.628,36.872,49.254
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.4_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__14-28-25_f2782a9801ba5ca2",35.580,63.089,44.165,54.144,46.213,67.084,33.922,49.171
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.1_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-06-01__10-35-39_d396de8867efad7b",34.812,62.584,44.214,51.776,45.957,66.757,35.960,48.866
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.3_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__13-36-00_10dc631133f2a445",36.348,63.258,43.009,52.565,45.650,67.301,33.534,48.809
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.05_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__16-05-33_eab70ba85008f4aa",34.215,61.406,42.412,51.223,44.626,67.247,35.048,48.025
"WD0.1_12Block_[0.5, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]_AlignedHidden0.1_NormEmbdsFalse_k=12_L=11_[[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11]]_2026-05-30__12-36-47_d396de8867efad7b",32.082,58.291,39.673,52.249,46.008,66.431,33.514,46.893


# Manually separating suits

In [294]:
convertion_dict = {
    "qa": [
        "arc:rc:bpb::olmes:full",
        "hellaswag:rc:bpb::olmes:full",
        "winogrande:rc:bpb::olmes:full",
        "socialiqa:rc:bpb::olmes:full",
        "piqa:rc:bpb::olmes:full",
        # "qasper_yesno:rc:bpb::olmes",
        "lambada:bpb",
    ],
    "math": [
        "minerva_math_algebra:bpb::olmes",
        "minerva_math_counting_and_probability:bpb::olmes",
        "minerva_math_geometry:bpb::olmes",
        "minerva_math_intermediate_algebra:bpb::olmes",
        "minerva_math_number_theory:bpb::olmes",
        "minerva_math_prealgebra:bpb::olmes",
        "minerva_math_precalculus:bpb::olmes",
    ],
    "code": [
        "codex_humaneval:3shot:bpb::modalities",
        "mbpp:3shot:bpb::modalities",
    ]
}

In [298]:
# Aggregate eval_full/* into the convertion_dict suites (qa / math / code) + overall average.
# Matching: an eval_full column is assigned to a suite if it matches a convertion_dict entry on
# BOTH the base task name AND the metric qualifiers (rc / bpb / 3shot / ...), after dropping the
# trailing dataset/source tag (everything from "::", e.g. ::olmes:full, ::none vs ::modalities).
# Requiring the qualifiers to match avoids pulling in the accuracy variant (e.g. piqa:rc::olmes:full)
# alongside the intended bpb variant (piqa:rc:bpb::olmes:full). The base-name prefix check still
# lets the "arc" entry expand to arc_challenge / arc_easy. Columns matching no entry are excluded.

RUN_ID = "y8qg2ips"
COL = "eval_full"
def aggregate_eval_full(run_id: str, project: str = PROJECT, entity: str = ENTITY, pct: bool = True):
    run = api.run(f"{entity}/{project}/{run_id}")
    hist = run.history(pandas=True, samples=100000000)
    cols = [c for c in hist.columns if c.startswith(f"{COL}/")]
    if not cols:
        raise ValueError(f"no {COL}/* columns for {run_id}")
    vals = hist.dropna(subset=cols, how="all")[cols].iloc[-1]
    if pct:
        vals = vals * 100

    def sig(n):
        left = n.split("::")[0]               # drop dataset/source tag (::olmes:full, ::none, ...)
        base, _, quals = left.partition(":")  # task name + ":"-joined metric qualifiers
        return base, quals

    suite_sigs = {s: [sig(t) for t in ts] for s, ts in convertion_dict.items()}

    def matches(cb, cq, eb, eq):
        # qualifiers must match exactly; base allows arc -> arc_challenge/arc_easy expansion
        return cq == eq and (cb.startswith(eb) or eb.startswith(cb))

    groups = {s: [] for s in convertion_dict}
    unmatched = []
    for c in cols:
        cb, cq = sig(c.replace(f"{COL}/", ""))
        hit = next((s for s, sigs in suite_sigs.items()
                    if any(matches(cb, cq, eb, eq) for eb, eq in sigs)), None)
        (groups[hit] if hit else unmatched).append(c)

    matched = [c for gc in groups.values() for c in gc]
    agg = {s: (vals[gc].mean() if gc else float("nan")) for s, gc in groups.items()}
    # agg["all"] = vals[matched].mean() if matched else float("nan")
    agg["all"] = sum([agg[s] for s in convertion_dict]) / len(convertion_dict)
    return pd.Series(agg, name=run.name), vals, groups, unmatched


agg, per_task, groups, unmatched = aggregate_eval_full(RUN_ID, pct=False)
pd.set_option("display.precision", 4)
print("Aggregates (%) via convertion_dict:")
for s, gc in groups.items():
    print(f"  {s:5s} (n={len(gc)}): {agg[s]:6.4f}   {[c.replace(f'{COL}/', '') for c in gc]}")
print(f"  {'ALL':5s} (n={sum(len(gc) for gc in groups.values())}): {agg['all']:6.4f}")
if unmatched:
    print("  unmatched (excluded):", [c.replace(f'{COL}/', '') for c in unmatched])


Aggregates (%) via convertion_dict:
  qa    (n=7): 1.0624   ['hellaswag:rc:bpb::olmes:full', 'piqa:rc:bpb::olmes:full', 'lambada:bpb', 'arc_easy:rc:bpb::olmes:full', 'arc_challenge:rc:bpb::olmes:full', 'winogrande:rc:bpb::olmes:full', 'socialiqa:rc:bpb::olmes:full']
  math  (n=7): 0.7765   ['minerva_math_number_theory:bpb::olmes', 'minerva_math_geometry:bpb::olmes', 'minerva_math_intermediate_algebra:bpb::olmes', 'minerva_math_algebra:bpb::olmes', 'minerva_math_counting_and_probability:bpb::olmes', 'minerva_math_prealgebra:bpb::olmes', 'minerva_math_precalculus:bpb::olmes']
  code  (n=2): 0.9653   ['mbpp:3shot:bpb::none', 'codex_humaneval:3shot:bpb::none']
  ALL   (n=16): 0.9348
  unmatched (excluded): ['modalities:base_easy:math_bpb', 'winogrande:rc::olmes:full', 'piqa:rc::olmes:full', 'socialiqa:rc::olmes:full', 'arc_easy:rc::olmes:full', 'gsm8k::olmes', 'olmo3:base_easy:math_bpb', 'lambada', 'arc_challenge:rc::olmes:full', 'hellaswag:rc::olmes:full', 'qasper_yesno:rc::olmes', 'modal